# AdLift - Ad Conversion Prediction
## DSAI Mini Coursework

**Student name**: *(fill in)*  
**Email**: *(fill in)*  
**Discord username**: *(fill in)*  
**Submission date**: *(fill in)*

---

### Mark Scheme

| Section | Marks |
|---|---|
| 1 | 10 |
| 2 | 3 |
| 3 | 10 |
| 4 | 20 |
| 5 | 15 |
| 6 | 8 |
| 7 | 19 |
| 8 | 5 |
| Extension | 10 |
| **Total** | **100** |

---
# SECTION 1 - Understanding the Problem *(10 marks)*

**1.1** *(2 marks)* What is the target variable? How do you construct it from the available tables? Is this a classification or regression problem?

*Your answer here.*

---

**1.2** *(2 marks)* Only ~15% of users convert. If a model simply predicts 'no conversion' for every user, what accuracy does it get? Why is this useless, and what metric should you use instead?

*Your answer here.*

---

**1.3** *(2 marks)* `was_clicked` in `ad_exposures` indicates that a user clicked an ad. Explain why click-through behaviour is a strong predictor of conversion. Is it possible that including it causes any form of data leakage?

*Your answer here.*

---

**1.4** *(2 marks)* The data contains some users who converted without ever being exposed to an ad (`ad_exposures.csv` has no record for them). What does this tell you about the nature of conversions in this dataset?

*Your answer here.*

---

**1.5** *(2 marks)* AdLift's budget allows targeting only the top 20% of users. In your own words, describe what the **conversion capture rate** measures and why it is a better business metric than AUC alone.

*Your answer here.*

---
# SECTION 2 - Setup & Data Loading *(3 marks)*

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

In [ ]:
# *(3 marks)* Load all four tables. Parse date columns. Print shape and first few rows.
users       = pd.read_csv('data/users.csv')
exposures   = pd.read_csv('data/ad_exposures.csv',  parse_dates=['exposure_date'])
conversions = pd.read_csv('data/conversions.csv',   parse_dates=['conversion_date'])
campaigns   = pd.read_csv('data/campaigns.csv',     parse_dates=['start_date','end_date'])

# Your code here

---
# SECTION 3 - Exploring the Data (EDA) *(10 marks)*

### 3.1 Data Quality & Target Construction *(3 marks)*

In [ ]:
# *(3 marks)*
# 1. Check nulls in each table.
# 2. Deduplicate ad_exposures on exposure_id.
# 3. Create the binary target column: users['converted'] = 1 if user_id in conversions.
# 4. Print the overall conversion rate.
# 5. How many users have no ad exposure at all?

# Your code here

**Commentary** - what did you find?

*Your answer here.*

### 3.2 Conversion Rate by User Segment *(3 marks)*

In [ ]:
# *(3 marks)* Plot conversion rate by:
# - age_group
# - is_loyalty_member
# - n_past_purchases (group into 0, 1, 2, 3+)
# - top_interest category

# Your code here

**Commentary** - which user characteristics show the strongest relationship with conversion?

*Your answer here.*

### 3.3 Ad Engagement Patterns *(4 marks)*

In [ ]:
# *(4 marks)* Explore:
# - Conversion rate of users who clicked >=1 ad vs those who didn't
# - Conversion rate by campaign (join to campaigns.csv for channel/cvr)
# - Distribution of ad exposures per user (how many ads do typical users see?)
# - Click rate overall and by channel

# Your code here

**Commentary** - which campaign/channel shows the highest conversion lift?

*Your answer here.*

---
# SECTION 4 - Feature Engineering *(20 marks)*

### 4.1 Aggregate ad exposures per user *(8 marks)*

In [ ]:
# *(8 marks)* Create a per-user exposure feature table. Include at least:
#
# Engagement:
#   - n_exposures: total ad impressions
#   - n_clicks: total clicks
#   - click_rate: n_clicks / n_exposures
#   - n_campaigns_seen: number of distinct campaigns
#   - avg_ad_position: mean position (lower = more prominent)
#
# Recency:
#   - days_since_last_exposure: how recently was the user last served an ad
#
# Channel breakdown:
#   - n_social_media, n_search, n_display, n_email, n_video
#
# Campaign quality:
#   - best_campaign_cvr: max(base_cvr) across campaigns the user was exposed to
#   - total_clicked_cvr: sum of base_cvr for campaigns where user clicked

# Your code here
# exp_feats = ...

### 4.2 Build master feature table *(8 marks)*

In [ ]:
# *(8 marks)* Join exp_feats to users.
# Encode categoricals:
#   - age_group: ordinal (18-24=0 ... 55+=4)
#   - gender: label encode
#   - top_interest: label encode or one-hot
#   - device_preference: label encode
# Fill NaN for users with no ad exposure (n_exposures=0, n_clicks=0, etc.)
# Print shape and null count of master.

# Your code here
# master = ...

***(4 marks)* Feature rationale.**

For each of your three most important features, explain what it measures and why it predicts conversion.
Are there any features that could cause leakage? Explain.

*Your answer here.*

---
# SECTION 5 - Training Models *(15 marks)*

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

TARGET  = 'converted'
EXCLUDE = ['user_id','age_group','gender','top_interest','device_preference', TARGET]

feature_cols = [c for c in master.columns if c not in EXCLUDE]
X = master[feature_cols]
y = master[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')
print(f'Conversion rate val: {y_val.mean():.2%}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

In [ ]:
# *(4 marks)* Model 1 - Logistic Regression
from sklearn.linear_model import LogisticRegression

# Your code here

***(3 marks)* Model 1 commentary.**

*Your answer here.*

In [ ]:
# *(4 marks)* Model 2 - Random Forest or LightGBM

# Your code here

***(4 marks)* Model 2 commentary.**

*Your answer here.*

In [ ]:
# Compare models - CV AUC, Val AUC, overfit gap.

# Your code here

best_prob  = None
best_model = None

**Which model did you choose and why?**

*Your answer here.*

---
# SECTION 6 - Evaluating Your Model *(8 marks)*

In [ ]:
# *(8 marks)* Produce:
# 1. ROC curves for both models
# 2. Score distributions (converted vs not converted)
# 3. Feature importance / coefficients (top 15)

# Your code here

**Interpret your diagnostics.**

*Your answer here.*

---
# SECTION 7 - Business Impact *(19 marks)*

In [ ]:
# *(10 marks)* Compute conversion capture rate at 20% targeting budget.
#
# Conversion capture rate = (# converters in top-20% scored) / (# total converters)
#
# Compare:
#   (a) Random 20% targeting
#   (b) Model top-20% targeting
#
# Plot the cumulative gain chart (% targeted vs % conversions captured).
#
# Assume: each targeted conversion generates £120 net revenue.
# Each targeted non-converter costs £3 in wasted ad spend.
# Compute net revenue for (a) random and (b) model targeting.

# Your code here

***(9 marks)* Business Recommendations.**

1. How many times better is model targeting vs random at the 20% budget?
2. At what % targeting rate does the model 'break even' vs random (same conversion capture rate as random at 20%)?
3. Which user segments should AdLift prioritise for high-value ad spend?
4. For users with low predicted conversion probability, what lower-cost touchpoints would you recommend?

*Your answer here.*

In [ ]:
import joblib
joblib.dump(best_model, 'model.pkl')
print('Model saved to model.pkl')

---
# SECTION 8 - Monitoring in Production *(5 marks)*

**8.1** *(2 marks)* AdLift runs new campaigns every month. When a new campaign is launched, some users will be exposed to it for the first time. How would this affect the feature distributions in your model, and how would you detect it?

*Your answer here.*

---

**8.2** *(3 marks)* The conversion rate can spike or drop dramatically around events like Black Friday, Christmas sales, or product launches. How would you monitor for this, and what would trigger a model retraining?

*Your answer here.*

---
# EXTENSION *(10 marks)*

Attempt **both** tasks (5 marks each).

**E1** *(5 marks)* **SHAP values**: For the top 50 highest-scoring users, use SHAP to identify what drives their predicted conversion probability. Are they dominated by loyalty membership, click behaviour, or campaign type? Plot a SHAP summary plot.

**E2** *(5 marks)* **Attribution**: A user may be exposed to 5 different campaigns before converting. Which campaign gets credit? Research multi-touch attribution models (last-click, linear, time-decay).